# MENG 417 — Homework #5
## Complete Step-by-Step Mathematical & Experimental Google Colab

**Source:** the one-page professor-provided **MENG 417 Homework #5**. All three questions are included, as are all four required subsections of Question 2. The final section can export a **single PDF with the source code appended**, as required by the professor.

| Professor's task | Coverage in this notebook |
|---|---|
| Q1. Tanh neuron with $\lambda=2$, two epochs, final weights and accuracy | Hand derivation, 8 sample updates, reproducible Python and independent gradient check |
| Q2. Three fully connected layers using Fashion-MNIST | Data preparation, 3-layer model, evaluation, confusion matrix, architecture, loss and training parameters |
| Q3. Five fully connected layers; compare using the same settings and observe GPU speed-up | 5-layer model, comparable accuracy and confusion matrices, explicit CPU-vs-GPU benchmark |
| Submission | One-PDF generator with code appendix, concise results and figures |

**Important: source constraints and transparent choices**

- Q1 specifies $\lambda=2$, squared error, starting $\mathbf w=(-1,1)^T$ and **two epochs**, but does **not** specify a learning rate, the exact definition of the parameterised tanh or whether updates are per-example or once per epoch. Therefore a unique numerical final weight vector **cannot be deduced from the handout alone**. We state and implement one standard convention and also show a second learning-rate example.
- Q2 and Q3 ask for real Fashion-MNIST test accuracy, confusion matrices and GPU time. These values **must be measured by executing the notebook**. They are never invented or prefilled.
- The input to the classifiers is 28×28 pixels, flattened to 784 features. There are exactly 3 and 5 **Dense** layers respectively; the input placeholder is not a Dense layer.

All displayed mathematics uses `$$...$$`.

# Question 1 — Tanh neuron, squared error and backpropagation

### Step 1 — Reconstruct the professor's four examples

The source gives four **ordered** training pairs:

$$
\begin{aligned}
\mathbf x^{(1)}&=(0,1)^T,&y^{(1)}&=+1,\\
\mathbf x^{(2)}&=(1,2)^T,&y^{(2)}&=+1,\\
\mathbf x^{(3)}&=(0,-1)^T,&y^{(3)}&=-1,\\
\mathbf x^{(4)}&=(-1,0)^T,&y^{(4)}&=-1.
\end{aligned}
$$

The neuron has **no bias term in the professor's equation**:

$$
\boxed{z=w_1x_1+w_2x_2=\mathbf w^T\mathbf x},\qquad
\boxed{\hat y=f_\lambda(z)}.
$$

The supplied starting weights are

$$
\boxed{\mathbf w^{(0)}=\begin{bmatrix}-1\\1\end{bmatrix}}.
$$

### Step 2 — Define the activation convention clearly

For the common *bipolar sigmoid* parameterisation,

$$
 f_\lambda(z)=\frac{1-e^{-\lambda z}}{1+e^{-\lambda z}}
 =\tanh\left(\frac{\lambda z}{2}\right).
$$

The professor specifies $\lambda=2$, so **under this declared convention**:

$$
\boxed{\hat y=\tanh(z)},\qquad
\boxed{f_\lambda'(z)=1-\hat y^2}.
$$

If the course defines $f_\lambda(z)=\tanh(\lambda z)$ instead, the resulting numerical updates will be different. The handout itself does not display an explicit functional formula, so this distinction must be kept visible.

### Step 3 — Derive the backpropagation update (every derivative)

Choose the common squared-error convention for **one** labelled example:

$$
\boxed{E=\frac12(y-\hat y)^2}.
$$

The factor $\tfrac12$ is a stated convention. If the course instead uses $E=(y-\hat y)^2$, the gradient is twice as large for the same learning rate.

Apply the chain rule:

$$
\frac{\partial E}{\partial\hat y}=\hat y-y,
\qquad
\frac{\partial\hat y}{\partial z}=1-\hat y^2,
\qquad
\frac{\partial z}{\partial w_j}=x_j.
$$

Thus

$$
\boxed{\frac{\partial E}{\partial w_j}
 =(\hat y-y)(1-\hat y^2)x_j}.
$$

In vector form, the **gradient-descent / backpropagation update** is

$$
\boxed{\mathbf w\leftarrow\mathbf w-
\eta(\hat y-y)(1-\hat y^2)\mathbf x}
$$

or equivalently

$$
\boxed{\mathbf w\leftarrow\mathbf w+
\eta(y-\hat y)(1-\hat y^2)\mathbf x}.
$$

### Step 4 — State the missing training choices

For the main worked example, we choose:

$$
\boxed{\eta=0.1},\qquad
\boxed{\text{online updates, in the professor's listed order}},
$$

with exactly **two passes** over the four examples: $2\times4=8$ updates. This is a legitimate reproducible example, **not** a claim that $\eta=0.1$ was supplied by the professor.

When classifying, use the sign of the final tanh output:

$$
\hat c=\begin{cases}+1,&\hat y\ge0,\\-1,&\hat y<0.\end{cases}
$$

In [ ]:
# HW5: Q1 — data and numerical implementation (no TensorFlow needed)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

X_Q1 = np.array([[0., 1.], [1., 2.], [0., -1.], [-1., 0.]])
Y_Q1 = np.array([1., 1., -1., -1.])
INITIAL_W = np.array([-1., 1.])
ETA_Q1 = 0.1         # Explicit assumption: NOT specified in source
EPOCHS_Q1 = 2       # Specified in source


def loss_single(w, x, target):
    yhat = np.tanh(np.dot(w, x))
    return 0.5 * (target - yhat)**2


def gradient_single(w, x, target):
    yhat = np.tanh(np.dot(w, x))
    return (yhat - target) * (1.0 - yhat**2) * x


def train_tanh_online(eta=ETA_Q1, epochs=2):
    w = INITIAL_W.copy()
    records = []
    for epoch in range(1, epochs+1):
        for index, (x, target) in enumerate(zip(X_Q1, Y_Q1), start=1):
            old_w = w.copy()
            z = float(old_w @ x)
            output = float(np.tanh(z))
            err = float(target - output)
            deriv = float(1.0 - output**2)
            delta = eta * err * deriv * x
            w += delta
            records.append(dict(epoch=epoch, sample=index, x=str(tuple(x)),
                                target=target, z=z, tanh=output,
                                error=err, derivative=deriv,
                                delta_w1=delta[0], delta_w2=delta[1],
                                w1=w[0], w2=w[1]))
    return w, pd.DataFrame(records)


def accuracy_tanh(w):
    outputs = np.tanh(X_Q1 @ w)
    preds = np.where(outputs >= 0.0, 1.0, -1.0)
    return float(np.mean(preds == Y_Q1)), outputs, preds

w_q1, trace_q1 = train_tanh_online()
acc_q1, outputs_q1, classes_q1 = accuracy_tanh(w_q1)
print('Question 1: all 8 per-example backpropagation updates:')
print(trace_q1.to_string(index=False, float_format=lambda a: f'{a:.9f}'))
print('Final weights:', w_q1)
print('Accuracy:', f'{100*acc_q1:.2f}%')
assert len(trace_q1) == 8

### Step 5 — Work the first update by hand

For sample 1, $\mathbf x=(0,1)^T$, $y=1$ and $\mathbf w^{(0)}=(-1,1)^T$:

$$
 z=(-1)(0)+(1)(1)=1.
$$

$$
\hat y=\tanh(1)\approx0.761594156.
$$

$$
 e=y-\hat y=1-0.761594156\approx0.238405844.
$$

$$
 f'(z)=1-\hat y^2\approx0.419974342.
$$

The weight correction for $\eta=0.1$ is

$$
 \Delta\mathbf w
 =0.1(0.238405844)(0.419974342)
 \begin{bmatrix}0\\1\end{bmatrix}
 \approx\begin{bmatrix}0\\0.010012434\end{bmatrix}.
$$

Therefore

$$
\boxed{\mathbf w^{(1)}\approx\begin{bmatrix}-1\\1.010012434\end{bmatrix}}.
$$

**For each of the seven remaining updates:** repeat exactly the same five operations: (1) dot product $z$, (2) tanh output, (3) error, (4) tanh derivative, (5) two-element weight correction. The previous cell prints the complete, reproducible values without intermediate rounding.

### Step 6 — Report the end of epoch 1 and end of epoch 2

Applying all four examples in order gives

$$
\boxed{\mathbf w_{\text{epoch 1}}\approx
\begin{bmatrix}-0.915752765\\1.037863066\end{bmatrix}}.
$$

After the second full pass,

$$
\boxed{\mathbf w_{\text{epoch 2}}\approx
\begin{bmatrix}-0.827683808\\1.065935974\end{bmatrix}}.
$$

Use the **final** vector to calculate every training score and class, not the initial weights. Under $\eta=0.1$ with this convention, three of the four examples are classified correctly:

$$
\boxed{\mathrm{Accuracy}=\frac34\times100\%=75\%}.
$$

This is the answer **for the explicitly stated learning rate and online update convention**. It is not a unique answer to the professor's wording, since changing $\eta$ can change the two-epoch result.

In [ ]:
# HW5: Q1 — final point-by-point verification
q1_verification = pd.DataFrame({
    'x1': X_Q1[:, 0], 'x2': X_Q1[:, 1], 'target': Y_Q1.astype(int),
    'score': X_Q1 @ w_q1, 'tanh output': outputs_q1,
    'predicted class': classes_q1.astype(int),
    'correct': classes_q1 == Y_Q1
})
print(q1_verification.to_string(index=False))
assert np.allclose(w_q1, [-0.8276838079889179, 1.0659359735175187], atol=1e-10)
assert abs(acc_q1 - 0.75) < 1e-12

# Gradient check: finite differences must agree with chain-rule gradient.
x0, y0 = X_Q1[1], Y_Q1[1]
step = 1e-6
numeric_gradient = np.array([
  (loss_single(INITIAL_W + step * np.eye(2)[j], x0, y0)
   - loss_single(INITIAL_W - step * np.eye(2)[j], x0, y0)) / (2*step)
  for j in range(2)
])
analytic_gradient = gradient_single(INITIAL_W, x0, y0)
print('Numerical gradient:', numeric_gradient)
print('Analytic gradient :', analytic_gradient)
assert np.allclose(numeric_gradient, analytic_gradient, atol=1e-8)

### Step 7 — Sensitivity to the learning rate (important for grading accuracy)

The professor does not specify $\eta$. For instance, with the **same activation, loss and online update order**, choosing $\eta=1$ instead of $0.1$ produces a different final vector and classification accuracy after two epochs.

This **does not mean** one set of numbers is universally correct. It demonstrates that an exact numerical answer requires the missing convention.

A further alternative is **batch gradient descent**, which aggregates the four gradients before each epoch update. That also produces different final weights and is not interchangeable with sequential (online) updates.

In [ ]:
# HW5: Q1 — show why the omitted learning rate matters
for eta in [0.1, 1.0]:
    w_eta, _ = train_tanh_online(eta=eta, epochs=2)
    acc_eta, _, _ = accuracy_tanh(w_eta)
    print(f'Online SGD, eta={eta}: w={w_eta}, accuracy={100*acc_eta:.1f}%')

# A clearly distinguished BATCH alternative for comparison.
def train_tanh_batch(eta=0.1, epochs=2):
    w = INITIAL_W.copy()
    for _ in range(epochs):
        out = np.tanh(X_Q1 @ w)
        grad = X_Q1.T @ ((out - Y_Q1) * (1-out**2)) / len(Y_Q1)
        w = w - eta * grad
    return w

wb = train_tanh_batch()
print('Batch mean-squared-error gradient, eta=0.1:', wb)

# Question 2 — Three-Dense-layer Fashion-MNIST classifier

## Step 1 — Identify the dataset and classification task

Fashion-MNIST contains ten types of clothing and footwear. Each image contains

$$
28\times28=784\quad\text{grayscale pixel intensities}.
$$

The target label belongs to

$$
y\in\{0,1,\ldots,9\}.
$$

The ten classes in the official dataset are T-shirt/top, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag and Ankle boot.

The professor **allows TensorFlow-Keras**. Unlike Homework #4, this handout does **not** forbid the official Fashion-MNIST Keras loader. We therefore use

```python
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
```

For a valid experiment:

1. Keep the official 10,000 test examples untouched until evaluation.
2. Convert raw `uint8` images to `float32` and divide by 255.
3. Flatten images into 784-length vectors.
4. Reserve 10% of the training set for validation, using one reproducible split shared by both networks.
5. Train both networks on the **same samples**.

All parameters below are our explicitly documented **design choices**, as the professor asks the student to design and report them.

In [ ]:
# HW5: Q2/Q3 — TensorFlow environment and reproducibility
import os
import time
import platform
from pathlib import Path

try:
    import tensorflow as tf
except ImportError as exc:
    raise ImportError('TensorFlow is required. In Google Colab use Runtime > Run all.') from exc

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

SEED = 417
EPOCHS = 12
BATCH_SIZE = 128
LEARNING_RATE = 0.001
VAL_FRACTION = 0.10

np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
print('TensorFlow:', tf.__version__)
print('Python:', platform.python_version())
print('GPUs visible:', tf.config.list_physical_devices('GPU'))
print('Training config: Adam lr=0.001, batch=128, epochs=12, 10% validation')

In [ ]:
# HW5: Q2 — load and preprocess the ORIGINAL official Fashion-MNIST data
(x_train_full, y_train_full), (x_test_raw, y_test) =     tf.keras.datasets.fashion_mnist.load_data()

assert x_train_full.shape == (60000, 28, 28)
assert x_test_raw.shape == (10000, 28, 28)
assert y_train_full.shape == (60000,)
assert y_test.shape == (10000,)

x_all = x_train_full.astype(np.float32).reshape(-1, 784) / 255.0
x_test = x_test_raw.astype(np.float32).reshape(-1, 784) / 255.0
y_train_full = y_train_full.astype(np.int64)
y_test = y_test.astype(np.int64)

x_train, x_val, y_train, y_val = train_test_split(
    x_all, y_train_full,
    test_size=VAL_FRACTION, random_state=SEED,
    stratify=y_train_full
)

CLASS_NAMES = ['T-shirt/top','Trouser','Pullover','Dress','Coat',
               'Sandal','Shirt','Sneaker','Bag','Ankle boot']
print('train', x_train.shape, 'validation', x_val.shape,
      'test', x_test.shape)
print('Pixel range:', float(x_train.min()), float(x_train.max()))
assert x_train.shape == (54000, 784)
assert x_val.shape == (6000, 784)
assert x_test.shape == (10000, 784)

In [ ]:
# HW5: Q2 — visualize representative training examples
fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for digit, ax in enumerate(axes.flat):
    k = int(np.where(y_train == digit)[0][0])
    ax.imshow(x_train[k].reshape(28,28), cmap='gray')
    ax.set_title(f'{digit}: {CLASS_NAMES[digit]}', fontsize=9)
    ax.axis('off')
fig.suptitle('Fashion-MNIST — one example of each class')
plt.tight_layout()
plt.show()

## Step 2 — Design exactly three fully connected layers

We choose this architecture:

$$
\boxed{784\longrightarrow128\;\mathrm{ReLU}
\longrightarrow64\;\mathrm{ReLU}
\longrightarrow10\;\mathrm{Softmax}}.
$$

There are **exactly three Dense layers**:

- Dense layer 1: 128 ReLU neurons.
- Dense layer 2: 64 ReLU neurons.
- Dense layer 3: 10 Softmax neurons.

The input specification is not counted as a Dense layer.

### Forward propagation through all three layers

$$
\begin{aligned}
\mathbf z^{(1)}&=W^{(1)}\mathbf x+\mathbf b^{(1)},
&\mathbf a^{(1)}&=\operatorname{ReLU}(\mathbf z^{(1)}),\\
\mathbf z^{(2)}&=W^{(2)}\mathbf a^{(1)}+\mathbf b^{(2)},
&\mathbf a^{(2)}&=\operatorname{ReLU}(\mathbf z^{(2)}),\\
\mathbf z^{(3)}&=W^{(3)}\mathbf a^{(2)}+\mathbf b^{(3)},
&p_k&=\frac{e^{z_k^{(3)}}}{\sum_{j=0}^9 e^{z_j^{(3)}}}.
\end{aligned}
$$

The predicted class is

$$
\boxed{\hat c=\arg\max_{k=0,\ldots,9}p_k}.
$$

### Exact parameter-count calculation

For a Dense layer with $d_{\rm in}$ inputs and $d_{\rm out}$ neurons,

$$
\boxed{P=(d_{\rm in}+1)d_{\rm out}}
$$

(the extra $1$ represents one bias per output neuron).

$$
\begin{aligned}
P_1&=(784+1)(128)=100{,}480,\\
P_2&=(128+1)(64)=8{,}256,\\
P_3&=(64+1)(10)=650.
\end{aligned}
$$

Therefore,

$$
\boxed{P_{\rm 3Dense}=100{,}480+8{,}256+650=109{,}386}.
$$

## Step 3 — Choose the loss function and optimiser

For a ten-class task, use **sparse categorical cross-entropy**. For one example with integer class $y$, the loss is

$$
\boxed{L=-\log p_y}.
$$

For $N$ training examples:

$$
\boxed{J=-\frac1N\sum_{i=1}^N \log p_{i,y_i}}.
$$

This is appropriate because the correct answer is **one class out of ten**; it measures how much probability the model places on that correct class.

**Adam optimiser**, initial learning rate $\eta=0.001$, batch size $128$ and $12$ epochs are documented choices. For ReLU hidden units, we use **He-normal initialisation**. No Dropout is applied, so the comparison with Question 3 has matching regularisation assumptions.

### Backpropagation intuition and central gradient

For the combination of Softmax and categorical cross-entropy, the derivative with respect to the ten output logits is

$$
\boxed{\frac{\partial L}{\partial z_k^{(3)}}=p_k-\mathbb{1}\{k=y\}}.
$$

This error propagates backward through the Dense matrices and the ReLU derivative

$$
\operatorname{ReLU}'(z)=\begin{cases}1,&z>0,\\0,&z<0.\end{cases}
$$

The gradients are then used by Adam to adjust the trainable weights and biases. TensorFlow automatically computes these derivatives and uses numerically stable implementations.

In [ ]:
# HW5: Q2/Q3 — EXACTLY three Dense vs five Dense layers

def make_dense_model(hidden_units, name='fashion_classifier'):
    model = tf.keras.Sequential(name=name)
    model.add(tf.keras.Input(shape=(784,)))
    for units in hidden_units:
        model.add(tf.keras.layers.Dense(
            units, activation='relu',
            kernel_initializer='he_normal'
        ))
    model.add(tf.keras.layers.Dense(10, activation='softmax'))
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=['accuracy']
    )
    return model

model_3 = make_dense_model((128, 64), name='fashion_3_dense')
model_3.summary()
assert len([l for l in model_3.layers if isinstance(l, tf.keras.layers.Dense)]) == 3
assert model_3.count_params() == 109386
print('Q2 three-Dense-layer parameter count:', model_3.count_params())

## Step 4 — Train the three-layer model

Training proceeds in mini-batches of 128 images. Each update performs:

1. Forward propagation through 128, 64 and 10 neurons.
2. Softmax probabilities for the ten labels.
3. Sparse cross-entropy loss for the batch.
4. Backpropagation of the loss gradients.
5. One Adam parameter update.
6. Validation assessment after each epoch, without fitting to validation examples.

The loss and accuracy curves below reveal underfitting or overfitting: high training accuracy but deteriorating validation loss can indicate overfitting.

In [ ]:
# HW5: Q2 — train once; DO NOT fabricate test results
start_3 = time.perf_counter()
history_3 = model_3.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    shuffle=True, verbose=2
)
time_3 = time.perf_counter() - start_3
print(f'Q2 3-Dense training wall time: {time_3:.2f} s')

## Step 5 — Measure test accuracy and construct the 10×10 confusion matrix

The test dataset contains 10,000 examples and must not be used to train either model.

$$
\boxed{\mathrm{Accuracy}=\frac{\#\text{correct test predictions}}{10000}\times100\%}.
$$

The confusion matrix has rows for **actual** labels and columns for **predicted** labels:

$$
\boxed{C_{i,j}=\#\{n:y_n=i,\ \hat y_n=j\}}.
$$

Its diagonal sums to the correctly classified test images. The cell below computes and validates both measurements.

In [ ]:
# HW5: Q2 — official test evaluation and confusion matrix
loss_3, accuracy_3 = model_3.evaluate(x_test, y_test, verbose=0)
pred_proba_3 = model_3.predict(x_test, batch_size=512, verbose=0)
pred_3 = np.argmax(pred_proba_3, axis=1)
cm_3 = confusion_matrix(y_test, pred_3, labels=np.arange(10))
assert cm_3.shape == (10,10) and cm_3.sum() == 10000
assert np.isclose(np.trace(cm_3) / 10000.0, accuracy_3, atol=1e-6)
print(f'Q2 TEST ACCURACY: {accuracy_3*100:.2f}%')
print(f'Q2 TEST LOSS: {loss_3:.6f}')
print('Q2 CORRECT TEST IMAGES:', np.trace(cm_3), '/ 10000')
print('Q2 CONFUSION MATRIX, rows=true labels, columns=predictions:')
print(pd.DataFrame(cm_3, index=CLASS_NAMES, columns=CLASS_NAMES).to_string())

In [ ]:
# HW5: Q2 — labelled 10x10 confusion-matrix figure

def draw_confusion_matrix(cm, title):
    fig, ax = plt.subplots(figsize=(10, 8))
    image = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(10), CLASS_NAMES, rotation=45, ha='right')
    ax.set_yticks(range(10), CLASS_NAMES)
    ax.set_xlabel('Predicted class')
    ax.set_ylabel('True class')
    ax.set_title(title)
    for i in range(10):
        for j in range(10):
            ax.text(j, i, str(cm[i,j]), ha='center', va='center',
                    fontsize=8, color='white' if cm[i,j] > cm.max()/2 else 'black')
    fig.colorbar(image, ax=ax, fraction=0.044, pad=0.04)
    fig.tight_layout()
    return fig

fig_cm3 = draw_confusion_matrix(cm_3, 'Question 2 — 3-Dense-layer Fashion-MNIST Confusion Matrix')
plt.show()

In [ ]:
# HW5: Q2 — training/validation learning curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(history_3.history['loss'], marker='o', label='training')
axes[0].plot(history_3.history['val_loss'], marker='o', label='validation')
axes[0].set_title('3 Dense layers — cross-entropy')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[1].plot(history_3.history['accuracy'], marker='o', label='training')
axes[1].plot(history_3.history['val_accuracy'], marker='o', label='validation')
axes[1].set_title('3 Dense layers — accuracy')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
for ax in axes:
    ax.grid(True, alpha=0.3)
    ax.legend()
fig.tight_layout(); plt.show()

# Question 3 — Expand to exactly five fully connected layers

## Step 1 — Preserve the Q2 training configuration

The professor says to increase the number of connected layers to five while using the **same** loss, parameters and optimiser. We therefore keep the **same**:

- Fashion-MNIST training, validation and official test observations;
- `SparseCategoricalCrossentropy`;
- Adam learning rate $0.001$;
- batch size $128$;
- number of epochs $12$;
- ReLU in hidden layers, Softmax for ten classes;
- He-normal initialisation and no Dropout.

Only the **network depth/width** is changed, since increasing to five Dense layers requires choosing two additional hidden layers.

### Architecture for five Dense layers

$$
\boxed{784\to128\;\mathrm{ReLU}
\to96\;\mathrm{ReLU}
\to64\;\mathrm{ReLU}
\to32\;\mathrm{ReLU}
\to10\;\mathrm{Softmax}}.
$$

This is **five** Dense layers, of which four are hidden and one is the output.

### Exact parameter count

$$
\begin{aligned}
P_1&=785(128)=100{,}480,\\
P_2&=129(96)=12{,}384,\\
P_3&=97(64)=6{,}208,\\
P_4&=65(32)=2{,}080,\\
P_5&=33(10)=330.
\end{aligned}
$$

Hence

$$
\boxed{P_{\rm 5Dense}=121{,}482}.
$$

**Fair comparison caution:** A 5-layer network has both more depth and a different number of parameters. Differences in accuracy should not be attributed solely to depth without additional controlled experiments. This is a correct comparison of the professor-requested *models* under fixed optimisation settings, not a causal proof that depth alone helps.

In [ ]:
# HW5: Q3 — five Dense layers with identical loss, optimiser & training hyperparameters
model_5 = make_dense_model((128,96,64,32), name='fashion_5_dense')
model_5.summary()
assert len([l for l in model_5.layers if isinstance(l, tf.keras.layers.Dense)]) == 5
assert model_5.count_params() == 121482
assert type(model_3.optimizer) == type(model_5.optimizer)
assert model_3.loss.__class__ == model_5.loss.__class__
print('Q3 five-Dense parameter count:', model_5.count_params())

In [ ]:
# HW5: Q3 — train five-Dense-layer network on the SAME split
start_5 = time.perf_counter()
history_5 = model_5.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    shuffle=True, verbose=2
)
time_5 = time.perf_counter() - start_5
print(f'Q3 5-Dense training wall time: {time_5:.2f} s')

In [ ]:
# HW5: Q3 — evaluate once on the independent official test set
loss_5, accuracy_5 = model_5.evaluate(x_test, y_test, verbose=0)
pred_proba_5 = model_5.predict(x_test, batch_size=512, verbose=0)
pred_5 = np.argmax(pred_proba_5, axis=1)
cm_5 = confusion_matrix(y_test, pred_5, labels=np.arange(10))
assert cm_5.sum() == 10000
assert np.isclose(np.trace(cm_5)/10000, accuracy_5, atol=1e-6)
print(f'Q3 TEST ACCURACY: {accuracy_5*100:.2f}%')
print(f'Q3 TEST LOSS: {loss_5:.6f}')
print('Q3 CONFUSION MATRIX:')
print(pd.DataFrame(cm_5, index=CLASS_NAMES, columns=CLASS_NAMES).to_string())
fig_cm5 = draw_confusion_matrix(cm_5, 'Question 3 — 5-Dense-layer Fashion-MNIST Confusion Matrix')
plt.show()

## Step 2 — Answer the professor's comparison question using measured results

Define the percentage-point change in accuracy as

$$
\boxed{\Delta_{\rm pp}=100(A_5-A_3)}.
$$

**If $\Delta_{\rm pp}>0$:** the five-layer network achieved higher accuracy in this run. A plausible mechanism is its increased capacity to learn nonlinear, hierarchical representations; however, this single comparison does **not** prove that depth caused the improvement.

**If $\Delta_{\rm pp}\le0$:** accuracy did not improve. Extra layers can make optimisation harder, add variance and overfitting risk, and provide no guaranteed improvement on a relatively small task such as Fashion-MNIST.

Always describe what your actual experiment shows. No expected accuracy is fabricated in this notebook.

In [ ]:
# HW5: Q3 — answer automatically from REAL runs
accuracy_diff_pp = 100.0 * (accuracy_5 - accuracy_3)
comparison = pd.DataFrame([
    dict(model='3 Dense', dense_layers=3, params=model_3.count_params(),
         test_accuracy_pct=100*accuracy_3, test_loss=loss_3,
         training_wall_seconds=time_3),
    dict(model='5 Dense', dense_layers=5, params=model_5.count_params(),
         test_accuracy_pct=100*accuracy_5, test_loss=loss_5,
         training_wall_seconds=time_5)
])
print(comparison.to_string(index=False))
print(f'Five minus three test accuracy = {accuracy_diff_pp:+.3f} percentage points')
if accuracy_diff_pp > 0:
    accuracy_conclusion = ('The 5-Dense-layer model improved accuracy in this run. '
                           'Extra representational capacity is a possible explanation; '
                           'this experiment alone does not isolate depth causally.')
elif accuracy_diff_pp < 0:
    accuracy_conclusion = ('The 5-Dense-layer model did not improve accuracy in this run. '
                           'More depth does not guarantee better generalisation.')
else:
    accuracy_conclusion = ('Both models achieved the same test accuracy in this run.')
print('Q3 ANSWER:', accuracy_conclusion)

In [ ]:
# HW5: Q3 — side-by-side training and validation learning curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for hist, label in [(history_3,'3 Dense'), (history_5,'5 Dense')]:
    axes[0].plot(hist.history['val_loss'], marker='.', label=label)
    axes[1].plot(hist.history['val_accuracy'], marker='.', label=label)
axes[0].set_title('Validation cross-entropy by depth')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[1].set_title('Validation accuracy by depth')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
for a in axes: a.grid(True,alpha=.3); a.legend()
plt.tight_layout(); plt.show()

## Step 3 — Test the GPU speed-up properly

The professor asks to **use a GPU to observe speed-up**. To answer this accurately, compare the time to train **the same five-layer architecture** on CPU and GPU using the same number of images, number of epochs, batch size, optimiser, loss and identical starting weights.

Comparing the **3-layer CPU time** against the **5-layer GPU time** would confound hardware and architecture; it is not a valid measure of GPU speed-up.

We therefore run a compact benchmark with:

- the same five-layer model on CPU and GPU;
- exactly **8,192** training images;
- exactly **2 epochs** with batch size 128;
- identical initial weights, and `shuffle=False` for controlled data order;
- a synchronized forward computation before stopping the timer to ensure the GPU has completed its work.

Compute

$$
\boxed{\mathrm{Speedup}=\frac{T_{\rm CPU}}{T_{\rm GPU}}}.
$$

The expected direction is **not guaranteed**: for small dense networks, GPU launch/transfer overhead can make training slower. TensorFlow can use a GPU only when one is genuinely available. In Google Colab select **Runtime → Change runtime type → GPU** before running the notebook. <Link url="https://www.tensorflow.org/guide/gpu" title="TensorFlow official GPU guide"/>.

In [ ]:
# HW5: Q3 — run a controlled five-layer CPU vs GPU benchmark
GPUS = tf.config.list_physical_devices('GPU')
BENCHMARK_N = 8192
BENCHMARK_EPOCHS = 2
X_BENCH = x_train[:BENCHMARK_N]
Y_BENCH = y_train[:BENCHMARK_N]

# Capture weights once so both model instances have the same starting point.
# (The benchmark uses new models; it does not reuse already-trained models.)
benchmark_seed_model = make_dense_model((128,96,64,32), name='benchmark_seed')
BENCHMARK_INITIAL_WEIGHTS = [a.copy() for a in benchmark_seed_model.get_weights()]


def benchmark_device(device_name):
    with tf.device(device_name):
        benchmark_model = make_dense_model((128,96,64,32), name='benchmark')
        benchmark_model.set_weights(BENCHMARK_INITIAL_WEIGHTS)
        # Warm up the inference kernels before starting the timer.
        _ = benchmark_model(tf.convert_to_tensor(X_BENCH[:128]), training=False).numpy()
        start = time.perf_counter()
        benchmark_model.fit(X_BENCH, Y_BENCH, epochs=BENCHMARK_EPOCHS,
                            batch_size=BATCH_SIZE, shuffle=False, verbose=0)
        # Tensor-to-NumPy conversion synchronizes pending GPU work.
        _ = benchmark_model(tf.convert_to_tensor(X_BENCH[:128]), training=False).numpy()
        elapsed = time.perf_counter() - start
    del benchmark_model
    return elapsed

cpu_seconds = benchmark_device('/CPU:0')
print(f'Five-Dense CPU time, {BENCHMARK_EPOCHS} epochs: {cpu_seconds:.3f} seconds')
if GPUS:
    gpu_seconds = benchmark_device('/GPU:0')
    gpu_speedup = cpu_seconds / gpu_seconds
    print(f'Five-Dense GPU time, {BENCHMARK_EPOCHS} epochs: {gpu_seconds:.3f} seconds')
    print(f'Measured GPU speed-up: {gpu_speedup:.3f}x')
else:
    gpu_seconds = None
    gpu_speedup = None
    print('GPU is NOT available in this runtime. No speed-up value can be reported.')
    print('Select Runtime > Change runtime type > GPU, then rerun the notebook.')

### Step 4 — Interpret the GPU timing honestly

- If speed-up $>1$, the measured GPU run was faster in this benchmark.
- If speed-up $\le1$, this run **did not** show a speed improvement; that result is valid and can occur for smaller neural networks.
- If no GPU is detected, **do not claim a GPU speed-up**: switch the Colab runtime type to GPU and rerun the benchmark.
- Device timings vary with Colab hardware, background load, TensorFlow compilation/warm-up, dataset transfer, and batch size; these are measured observations, not immutable constants.

The correct final conclusion combines **two separate pieces of evidence**: (1) accuracy difference between the 3- and 5-layer networks on the same 10,000 test examples, and (2) CPU-vs-GPU speed-up of the same model and workload.

In [ ]:
# HW5: Q3 — produce professor-facing conclusion after measurement
if gpu_speedup is None:
    hardware_sentence = ('GPU benchmark: not completed because the current runtime lacks a visible GPU.')
else:
    comparison_word = 'faster' if gpu_speedup > 1 else 'not faster'
    hardware_sentence = (f'For the same five-layer model and two-epoch benchmark, '
                         f'CPU time={cpu_seconds:.3f}s and GPU time={gpu_seconds:.3f}s; '
                         f'GPU was {comparison_word} (speed-up={gpu_speedup:.3f}x).')
print('Q2: test accuracy =',f'{100*accuracy_3:.2f}%',
      'parameters =',model_3.count_params())
print('Q3: test accuracy =',f'{100*accuracy_5:.2f}%',
      'parameters =',model_5.count_params())
print(accuracy_conclusion)
print(hardware_sentence)

# All-question answer summary (automatically grounded in your run)

| Question | Required answer | Verification |
|---|---|---|
| **Q1** | Two-epoch tanh backpropagation, final weights, training accuracy | Exact 8-step table, finite-difference gradient check and final classification table |
| **Q2(a)** | Fashion-MNIST test accuracy and confusion matrix | Computed after actual training; rows are true labels and columns predicted labels |
| **Q2(b)** | Three Dense layer structure | $784\to128\to64\to10$, exactly 109,386 parameters |
| **Q2(c)** | Loss function | Sparse categorical cross-entropy $-\frac1N\sum_i\log p_{i,y_i}$ |
| **Q2(d)** | Training parameters and optimiser | Adam, $\eta=0.001$, batch 128, 12 epochs, ReLU, He-normal, 10% validation; no Dropout |
| **Q3** | Five Dense layers, same training configuration, accuracy comparison and GPU speed-up | $784\to128\to96\to64\to32\to10$, 121,482 parameters; measured test accuracy and matched CPU/GPU workload |

**Submission note:** The professor requires **one PDF with the code appended**. The next section generates this PDF only once the actual Q2/Q3 experiment has run; it copies the exact measured figures, configuration, conclusions, and reproducible source code into the report.

# End-of-homework theory cheat sheet

## Tanh and backpropagation

$$
\tanh z=\frac{e^z-e^{-z}}{e^z+e^{-z}},
\qquad \frac{d}{dz}\tanh z=1-\tanh^2z.
$$

$$
E=\frac12(y-\hat y)^2,
\qquad \boxed{\frac{\partial E}{\partial\mathbf w}
=(\hat y-y)(1-\hat y^2)\mathbf x}.
$$

**Epoch:** one complete pass through all four examples. In online SGD, there are four parameter updates per epoch; in batch gradient descent, there is one update per epoch.

## Dense layers and parameters

$$
\boxed{P_{\rm dense}=(n_{\rm input}+1)n_{\rm output}}.
$$

Increasing the number of Dense layers can increase representational capacity, but it does not guarantee higher accuracy.

## Softmax and cross-entropy

$$
p_k=\frac{e^{z_k}}{\sum_j e^{z_j}},
\qquad L=-\log p_y.
$$

**ReLU:** $\max(0,z)$, introduces nonlinearity into the network.

## Overfitting and validation

A model that performs better on training data but worse on held-out validation/test data may be overfitting. Keep test data out of architecture and hyperparameter selection.

## GPU benchmarking

$$
\boxed{\text{GPU speed-up}=\frac{\text{CPU time}}{\text{GPU time}}}.
$$

The comparison must use the **same model and workload** on both devices. A value less than one is possible.

## Frequent errors to avoid

1. Treating $\lambda=2$ as an automatically supplied learning rate in Q1.
2. Forgetting the derivative $1-\tanh^2(z)$ or the chain rule.
3. Adding an unrequested bias to the Q1 neuron without declaring it.
4. Counting the Keras input specification as a Dense layer.
5. Reporting test-set accuracy before the network has actually trained.
6. Changing the optimiser or learning rate between Q2 and Q3.
7. Comparing different network architectures on CPU and GPU and attributing the runtime difference solely to the hardware.
8. Claiming deeper networks always improve accuracy.
9. Submitting a report without appending the source code.

**Mental model:** examples → weights → neuron/network → prediction → error → backpropagation → optimiser update → validation → independent test evaluation.

# Generate the required single submission PDF (after running all previous cells)

The professor explicitly asks for **one PDF with code appended**. The following cell generates `MENG417_Homework5_Submission.pdf` using the results from your **actual run**, including both 10×10 confusion matrices and the complete executable Python source of the experiment.

**Before running it:** ensure Q1, Q2, Q3 and the GPU benchmark cells have completed successfully. If Colab has no GPU, the report will state that the hardware speed-up was not measured; this does **not** satisfy the professor's request until you rerun on a GPU.

The PDF uses readable mathematical equations, separate question headings and properly captioned graphs, followed by the full experiment source in smaller monospaced text.

In [ ]:
EXPERIMENT_SOURCE = '"""MENG 417 Homework #5 — executable experiment from the matching Colab.\nRequires: tensorflow, numpy, pandas, matplotlib, scikit-learn.\nFor Question 3 GPU timing, run on a GPU-capable TensorFlow environment.\n"""\n\n# %%\n# HW5: Q1 — data and numerical implementation (no TensorFlow needed)\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\nX_Q1 = np.array([[0., 1.], [1., 2.], [0., -1.], [-1., 0.]])\nY_Q1 = np.array([1., 1., -1., -1.])\nINITIAL_W = np.array([-1., 1.])\nETA_Q1 = 0.1         # Explicit assumption: NOT specified in source\nEPOCHS_Q1 = 2       # Specified in source\n\n\ndef loss_single(w, x, target):\n    yhat = np.tanh(np.dot(w, x))\n    return 0.5 * (target - yhat)**2\n\n\ndef gradient_single(w, x, target):\n    yhat = np.tanh(np.dot(w, x))\n    return (yhat - target) * (1.0 - yhat**2) * x\n\n\ndef train_tanh_online(eta=ETA_Q1, epochs=2):\n    w = INITIAL_W.copy()\n    records = []\n    for epoch in range(1, epochs+1):\n        for index, (x, target) in enumerate(zip(X_Q1, Y_Q1), start=1):\n            old_w = w.copy()\n            z = float(old_w @ x)\n            output = float(np.tanh(z))\n            err = float(target - output)\n            deriv = float(1.0 - output**2)\n            delta = eta * err * deriv * x\n            w += delta\n            records.append(dict(epoch=epoch, sample=index, x=str(tuple(x)),\n                                target=target, z=z, tanh=output,\n                                error=err, derivative=deriv,\n                                delta_w1=delta[0], delta_w2=delta[1],\n                                w1=w[0], w2=w[1]))\n    return w, pd.DataFrame(records)\n\n\ndef accuracy_tanh(w):\n    outputs = np.tanh(X_Q1 @ w)\n    preds = np.where(outputs >= 0.0, 1.0, -1.0)\n    return float(np.mean(preds == Y_Q1)), outputs, preds\n\nw_q1, trace_q1 = train_tanh_online()\nacc_q1, outputs_q1, classes_q1 = accuracy_tanh(w_q1)\nprint(\'Question 1: all 8 per-example backpropagation updates:\')\nprint(trace_q1.to_string(index=False, float_format=lambda a: f\'{a:.9f}\'))\nprint(\'Final weights:\', w_q1)\nprint(\'Accuracy:\', f\'{100*acc_q1:.2f}%\')\nassert len(trace_q1) == 8\n\n# %%\n# HW5: Q1 — final point-by-point verification\nq1_verification = pd.DataFrame({\n    \'x1\': X_Q1[:, 0], \'x2\': X_Q1[:, 1], \'target\': Y_Q1.astype(int),\n    \'score\': X_Q1 @ w_q1, \'tanh output\': outputs_q1,\n    \'predicted class\': classes_q1.astype(int),\n    \'correct\': classes_q1 == Y_Q1\n})\nprint(q1_verification.to_string(index=False))\nassert np.allclose(w_q1, [-0.8276838079889179, 1.0659359735175187], atol=1e-10)\nassert abs(acc_q1 - 0.75) < 1e-12\n\n# Gradient check: finite differences must agree with chain-rule gradient.\nx0, y0 = X_Q1[1], Y_Q1[1]\nstep = 1e-6\nnumeric_gradient = np.array([\n  (loss_single(INITIAL_W + step * np.eye(2)[j], x0, y0)\n   - loss_single(INITIAL_W - step * np.eye(2)[j], x0, y0)) / (2*step)\n  for j in range(2)\n])\nanalytic_gradient = gradient_single(INITIAL_W, x0, y0)\nprint(\'Numerical gradient:\', numeric_gradient)\nprint(\'Analytic gradient :\', analytic_gradient)\nassert np.allclose(numeric_gradient, analytic_gradient, atol=1e-8)\n\n# %%\n# HW5: Q1 — show why the omitted learning rate matters\nfor eta in [0.1, 1.0]:\n    w_eta, _ = train_tanh_online(eta=eta, epochs=2)\n    acc_eta, _, _ = accuracy_tanh(w_eta)\n    print(f\'Online SGD, eta={eta}: w={w_eta}, accuracy={100*acc_eta:.1f}%\')\n\n# A clearly distinguished BATCH alternative for comparison.\ndef train_tanh_batch(eta=0.1, epochs=2):\n    w = INITIAL_W.copy()\n    for _ in range(epochs):\n        out = np.tanh(X_Q1 @ w)\n        grad = X_Q1.T @ ((out - Y_Q1) * (1-out**2)) / len(Y_Q1)\n        w = w - eta * grad\n    return w\n\nwb = train_tanh_batch()\nprint(\'Batch mean-squared-error gradient, eta=0.1:\', wb)\n\n# %%\n# HW5: Q2/Q3 — TensorFlow environment and reproducibility\nimport os\nimport time\nimport platform\nfrom pathlib import Path\n\ntry:\n    import tensorflow as tf\nexcept ImportError as exc:\n    raise ImportError(\'TensorFlow is required. In Google Colab use Runtime > Run all.\') from exc\n\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.metrics import confusion_matrix, classification_report\n\nSEED = 417\nEPOCHS = 12\nBATCH_SIZE = 128\nLEARNING_RATE = 0.001\nVAL_FRACTION = 0.10\n\nnp.random.seed(SEED)\ntf.keras.utils.set_random_seed(SEED)\nprint(\'TensorFlow:\', tf.__version__)\nprint(\'Python:\', platform.python_version())\nprint(\'GPUs visible:\', tf.config.list_physical_devices(\'GPU\'))\nprint(\'Training config: Adam lr=0.001, batch=128, epochs=12, 10% validation\')\n\n# %%\n# HW5: Q2 — load and preprocess the ORIGINAL official Fashion-MNIST data\n(x_train_full, y_train_full), (x_test_raw, y_test) =     tf.keras.datasets.fashion_mnist.load_data()\n\nassert x_train_full.shape == (60000, 28, 28)\nassert x_test_raw.shape == (10000, 28, 28)\nassert y_train_full.shape == (60000,)\nassert y_test.shape == (10000,)\n\nx_all = x_train_full.astype(np.float32).reshape(-1, 784) / 255.0\nx_test = x_test_raw.astype(np.float32).reshape(-1, 784) / 255.0\ny_train_full = y_train_full.astype(np.int64)\ny_test = y_test.astype(np.int64)\n\nx_train, x_val, y_train, y_val = train_test_split(\n    x_all, y_train_full,\n    test_size=VAL_FRACTION, random_state=SEED,\n    stratify=y_train_full\n)\n\nCLASS_NAMES = [\'T-shirt/top\',\'Trouser\',\'Pullover\',\'Dress\',\'Coat\',\n               \'Sandal\',\'Shirt\',\'Sneaker\',\'Bag\',\'Ankle boot\']\nprint(\'train\', x_train.shape, \'validation\', x_val.shape,\n      \'test\', x_test.shape)\nprint(\'Pixel range:\', float(x_train.min()), float(x_train.max()))\nassert x_train.shape == (54000, 784)\nassert x_val.shape == (6000, 784)\nassert x_test.shape == (10000, 784)\n\n# %%\n# HW5: Q2 — visualize representative training examples\nfig, axes = plt.subplots(2, 5, figsize=(11, 5))\nfor digit, ax in enumerate(axes.flat):\n    k = int(np.where(y_train == digit)[0][0])\n    ax.imshow(x_train[k].reshape(28,28), cmap=\'gray\')\n    ax.set_title(f\'{digit}: {CLASS_NAMES[digit]}\', fontsize=9)\n    ax.axis(\'off\')\nfig.suptitle(\'Fashion-MNIST — one example of each class\')\nplt.tight_layout()\nplt.show()\n\n# %%\n# HW5: Q2/Q3 — EXACTLY three Dense vs five Dense layers\n\ndef make_dense_model(hidden_units, name=\'fashion_classifier\'):\n    model = tf.keras.Sequential(name=name)\n    model.add(tf.keras.Input(shape=(784,)))\n    for units in hidden_units:\n        model.add(tf.keras.layers.Dense(\n            units, activation=\'relu\',\n            kernel_initializer=\'he_normal\'\n        ))\n    model.add(tf.keras.layers.Dense(10, activation=\'softmax\'))\n    model.compile(\n        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),\n        loss=tf.keras.losses.SparseCategoricalCrossentropy(),\n        metrics=[\'accuracy\']\n    )\n    return model\n\nmodel_3 = make_dense_model((128, 64), name=\'fashion_3_dense\')\nmodel_3.summary()\nassert len([l for l in model_3.layers if isinstance(l, tf.keras.layers.Dense)]) == 3\nassert model_3.count_params() == 109386\nprint(\'Q2 three-Dense-layer parameter count:\', model_3.count_params())\n\n# %%\n# HW5: Q2 — train once; DO NOT fabricate test results\nstart_3 = time.perf_counter()\nhistory_3 = model_3.fit(\n    x_train, y_train,\n    validation_data=(x_val, y_val),\n    epochs=EPOCHS, batch_size=BATCH_SIZE,\n    shuffle=True, verbose=2\n)\ntime_3 = time.perf_counter() - start_3\nprint(f\'Q2 3-Dense training wall time: {time_3:.2f} s\')\n\n# %%\n# HW5: Q2 — official test evaluation and confusion matrix\nloss_3, accuracy_3 = model_3.evaluate(x_test, y_test, verbose=0)\npred_proba_3 = model_3.predict(x_test, batch_size=512, verbose=0)\npred_3 = np.argmax(pred_proba_3, axis=1)\ncm_3 = confusion_matrix(y_test, pred_3, labels=np.arange(10))\nassert cm_3.shape == (10,10) and cm_3.sum() == 10000\nassert np.isclose(np.trace(cm_3) / 10000.0, accuracy_3, atol=1e-6)\nprint(f\'Q2 TEST ACCURACY: {accuracy_3*100:.2f}%\')\nprint(f\'Q2 TEST LOSS: {loss_3:.6f}\')\nprint(\'Q2 CORRECT TEST IMAGES:\', np.trace(cm_3), \'/ 10000\')\nprint(\'Q2 CONFUSION MATRIX, rows=true labels, columns=predictions:\')\nprint(pd.DataFrame(cm_3, index=CLASS_NAMES, columns=CLASS_NAMES).to_string())\n\n# %%\n# HW5: Q2 — labelled 10x10 confusion-matrix figure\n\ndef draw_confusion_matrix(cm, title):\n    fig, ax = plt.subplots(figsize=(10, 8))\n    image = ax.imshow(cm, cmap=\'Blues\')\n    ax.set_xticks(range(10), CLASS_NAMES, rotation=45, ha=\'right\')\n    ax.set_yticks(range(10), CLASS_NAMES)\n    ax.set_xlabel(\'Predicted class\')\n    ax.set_ylabel(\'True class\')\n    ax.set_title(title)\n    for i in range(10):\n        for j in range(10):\n            ax.text(j, i, str(cm[i,j]), ha=\'center\', va=\'center\',\n                    fontsize=8, color=\'white\' if cm[i,j] > cm.max()/2 else \'black\')\n    fig.colorbar(image, ax=ax, fraction=0.044, pad=0.04)\n    fig.tight_layout()\n    return fig\n\nfig_cm3 = draw_confusion_matrix(cm_3, \'Question 2 — 3-Dense-layer Fashion-MNIST Confusion Matrix\')\nplt.show()\n\n# %%\n# HW5: Q2 — training/validation learning curves\nfig, axes = plt.subplots(1, 2, figsize=(12, 4.5))\naxes[0].plot(history_3.history[\'loss\'], marker=\'o\', label=\'training\')\naxes[0].plot(history_3.history[\'val_loss\'], marker=\'o\', label=\'validation\')\naxes[0].set_title(\'3 Dense layers — cross-entropy\')\naxes[0].set_xlabel(\'Epoch\'); axes[0].set_ylabel(\'Loss\')\naxes[1].plot(history_3.history[\'accuracy\'], marker=\'o\', label=\'training\')\naxes[1].plot(history_3.history[\'val_accuracy\'], marker=\'o\', label=\'validation\')\naxes[1].set_title(\'3 Dense layers — accuracy\')\naxes[1].set_xlabel(\'Epoch\'); axes[1].set_ylabel(\'Accuracy\')\nfor ax in axes:\n    ax.grid(True, alpha=0.3)\n    ax.legend()\nfig.tight_layout(); plt.show()\n\n# %%\n# HW5: Q3 — five Dense layers with identical loss, optimiser & training hyperparameters\nmodel_5 = make_dense_model((128,96,64,32), name=\'fashion_5_dense\')\nmodel_5.summary()\nassert len([l for l in model_5.layers if isinstance(l, tf.keras.layers.Dense)]) == 5\nassert model_5.count_params() == 121482\nassert type(model_3.optimizer) == type(model_5.optimizer)\nassert model_3.loss.__class__ == model_5.loss.__class__\nprint(\'Q3 five-Dense parameter count:\', model_5.count_params())\n\n# %%\n# HW5: Q3 — train five-Dense-layer network on the SAME split\nstart_5 = time.perf_counter()\nhistory_5 = model_5.fit(\n    x_train, y_train,\n    validation_data=(x_val, y_val),\n    epochs=EPOCHS, batch_size=BATCH_SIZE,\n    shuffle=True, verbose=2\n)\ntime_5 = time.perf_counter() - start_5\nprint(f\'Q3 5-Dense training wall time: {time_5:.2f} s\')\n\n# %%\n# HW5: Q3 — evaluate once on the independent official test set\nloss_5, accuracy_5 = model_5.evaluate(x_test, y_test, verbose=0)\npred_proba_5 = model_5.predict(x_test, batch_size=512, verbose=0)\npred_5 = np.argmax(pred_proba_5, axis=1)\ncm_5 = confusion_matrix(y_test, pred_5, labels=np.arange(10))\nassert cm_5.sum() == 10000\nassert np.isclose(np.trace(cm_5)/10000, accuracy_5, atol=1e-6)\nprint(f\'Q3 TEST ACCURACY: {accuracy_5*100:.2f}%\')\nprint(f\'Q3 TEST LOSS: {loss_5:.6f}\')\nprint(\'Q3 CONFUSION MATRIX:\')\nprint(pd.DataFrame(cm_5, index=CLASS_NAMES, columns=CLASS_NAMES).to_string())\nfig_cm5 = draw_confusion_matrix(cm_5, \'Question 3 — 5-Dense-layer Fashion-MNIST Confusion Matrix\')\nplt.show()\n\n# %%\n# HW5: Q3 — answer automatically from REAL runs\naccuracy_diff_pp = 100.0 * (accuracy_5 - accuracy_3)\ncomparison = pd.DataFrame([\n    dict(model=\'3 Dense\', dense_layers=3, params=model_3.count_params(),\n         test_accuracy_pct=100*accuracy_3, test_loss=loss_3,\n         training_wall_seconds=time_3),\n    dict(model=\'5 Dense\', dense_layers=5, params=model_5.count_params(),\n         test_accuracy_pct=100*accuracy_5, test_loss=loss_5,\n         training_wall_seconds=time_5)\n])\nprint(comparison.to_string(index=False))\nprint(f\'Five minus three test accuracy = {accuracy_diff_pp:+.3f} percentage points\')\nif accuracy_diff_pp > 0:\n    accuracy_conclusion = (\'The 5-Dense-layer model improved accuracy in this run. \'\n                           \'Extra representational capacity is a possible explanation; \'\n                           \'this experiment alone does not isolate depth causally.\')\nelif accuracy_diff_pp < 0:\n    accuracy_conclusion = (\'The 5-Dense-layer model did not improve accuracy in this run. \'\n                           \'More depth does not guarantee better generalisation.\')\nelse:\n    accuracy_conclusion = (\'Both models achieved the same test accuracy in this run.\')\nprint(\'Q3 ANSWER:\', accuracy_conclusion)\n\n# %%\n# HW5: Q3 — side-by-side training and validation learning curves\nfig, axes = plt.subplots(1, 2, figsize=(12, 4.5))\nfor hist, label in [(history_3,\'3 Dense\'), (history_5,\'5 Dense\')]:\n    axes[0].plot(hist.history[\'val_loss\'], marker=\'.\', label=label)\n    axes[1].plot(hist.history[\'val_accuracy\'], marker=\'.\', label=label)\naxes[0].set_title(\'Validation cross-entropy by depth\')\naxes[0].set_xlabel(\'Epoch\'); axes[0].set_ylabel(\'Loss\')\naxes[1].set_title(\'Validation accuracy by depth\')\naxes[1].set_xlabel(\'Epoch\'); axes[1].set_ylabel(\'Accuracy\')\nfor a in axes: a.grid(True,alpha=.3); a.legend()\nplt.tight_layout(); plt.show()\n\n# %%\n# HW5: Q3 — run a controlled five-layer CPU vs GPU benchmark\nGPUS = tf.config.list_physical_devices(\'GPU\')\nBENCHMARK_N = 8192\nBENCHMARK_EPOCHS = 2\nX_BENCH = x_train[:BENCHMARK_N]\nY_BENCH = y_train[:BENCHMARK_N]\n\n# Capture weights once so both model instances have the same starting point.\n# (The benchmark uses new models; it does not reuse already-trained models.)\nbenchmark_seed_model = make_dense_model((128,96,64,32), name=\'benchmark_seed\')\nBENCHMARK_INITIAL_WEIGHTS = [a.copy() for a in benchmark_seed_model.get_weights()]\n\n\ndef benchmark_device(device_name):\n    with tf.device(device_name):\n        benchmark_model = make_dense_model((128,96,64,32), name=\'benchmark\')\n        benchmark_model.set_weights(BENCHMARK_INITIAL_WEIGHTS)\n        # Warm up the inference kernels before starting the timer.\n        _ = benchmark_model(tf.convert_to_tensor(X_BENCH[:128]), training=False).numpy()\n        start = time.perf_counter()\n        benchmark_model.fit(X_BENCH, Y_BENCH, epochs=BENCHMARK_EPOCHS,\n                            batch_size=BATCH_SIZE, shuffle=False, verbose=0)\n        # Tensor-to-NumPy conversion synchronizes pending GPU work.\n        _ = benchmark_model(tf.convert_to_tensor(X_BENCH[:128]), training=False).numpy()\n        elapsed = time.perf_counter() - start\n    del benchmark_model\n    return elapsed\n\ncpu_seconds = benchmark_device(\'/CPU:0\')\nprint(f\'Five-Dense CPU time, {BENCHMARK_EPOCHS} epochs: {cpu_seconds:.3f} seconds\')\nif GPUS:\n    gpu_seconds = benchmark_device(\'/GPU:0\')\n    gpu_speedup = cpu_seconds / gpu_seconds\n    print(f\'Five-Dense GPU time, {BENCHMARK_EPOCHS} epochs: {gpu_seconds:.3f} seconds\')\n    print(f\'Measured GPU speed-up: {gpu_speedup:.3f}x\')\nelse:\n    gpu_seconds = None\n    gpu_speedup = None\n    print(\'GPU is NOT available in this runtime. No speed-up value can be reported.\')\n    print(\'Select Runtime > Change runtime type > GPU, then rerun the notebook.\')\n\n# %%\n# HW5: Q3 — produce professor-facing conclusion after measurement\nif gpu_speedup is None:\n    hardware_sentence = (\'GPU benchmark: not completed because the current runtime lacks a visible GPU.\')\nelse:\n    comparison_word = \'faster\' if gpu_speedup > 1 else \'not faster\'\n    hardware_sentence = (f\'For the same five-layer model and two-epoch benchmark, \'\n                         f\'CPU time={cpu_seconds:.3f}s and GPU time={gpu_seconds:.3f}s; \'\n                         f\'GPU was {comparison_word} (speed-up={gpu_speedup:.3f}x).\')\nprint(\'Q2: test accuracy =\',f\'{100*accuracy_3:.2f}%\',\n      \'parameters =\',model_3.count_params())\nprint(\'Q3: test accuracy =\',f\'{100*accuracy_5:.2f}%\',\n      \'parameters =\',model_5.count_params())\nprint(accuracy_conclusion)\nprint(hardware_sentence)\n'
with open("MENG417_Homework5_Experiment.py", "w", encoding="utf-8") as f:
    f.write(EXPERIMENT_SOURCE)
print("Wrote complete Python source appendix: MENG417_Homework5_Experiment.py")

In [ ]:
# PDF generator — based on actual experiment values only; never inserts invented metrics.
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer, Image,
                                 Table, TableStyle, PageBreak, Preformatted, KeepTogether)
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER
from reportlab.lib.units import mm
from reportlab.pdfbase.pdfmetrics import stringWidth
from PIL import Image as PILImage
import textwrap

REPORT_PATH = 'MENG417_Homework5_Submission.pdf'
styles = getSampleStyleSheet()
styles.add(ParagraphStyle(name='HW5Title', parent=styles['Title'],
                          alignment=TA_CENTER,fontSize=18,leading=22,spaceAfter=12))
styles.add(ParagraphStyle(name='HW5Head', parent=styles['Heading2'],
                          fontSize=12.5,leading=16,spaceBefore=12,spaceAfter=6))
styles.add(ParagraphStyle(name='HW5Body',parent=styles['BodyText'],
                          fontSize=9.3,leading=13,spaceAfter=7))
styles.add(ParagraphStyle(name='HW5Small',parent=styles['BodyText'],
                          fontSize=8,leading=11,spaceAfter=6))

# Save the figures needed in the PDF.
fig_cm3.savefig('HW5_CM_3Dense.png',dpi=145,bbox_inches='tight')
fig_cm5.savefig('HW5_CM_5Dense.png',dpi=145,bbox_inches='tight')
fig, ax=plt.subplots(figsize=(7,4))
ax.bar(['3 Dense','5 Dense'],[100*accuracy_3,100*accuracy_5])
ax.set_ylabel('Official test accuracy (%)')
ax.set_title('Same Fashion-MNIST test set, 10,000 images')
ax.set_ylim(max(0,min(100*accuracy_3,100*accuracy_5)-8),
            min(100,max(100*accuracy_3,100*accuracy_5)+4))
fig.tight_layout()
fig.savefig('HW5_Accuracy_Comparison.png',dpi=145,bbox_inches='tight')
plt.close(fig)


def add_figure(file, width=150*mm):
    with PILImage.open(file) as im:
        w,h=im.size
    return Image(file,width=width,height=width*h/w)


def P(s,style='HW5Body'):
    return Paragraph(s,styles[style])


def eq_image(latex, fname):
    # Matplotlib renders a true mathematical formula into a legible image.
    fig=plt.figure(figsize=(7.5,.64))
    fig.patch.set_facecolor('white')
    fig.text(.5,.5,'$'+latex+'$',ha='center',va='center',fontsize=17)
    fig.savefig(fname,dpi=220,bbox_inches='tight',pad_inches=.08)
    plt.close(fig)
    return add_figure(fname,width=120*mm)

story=[
    P('MENG 417 — Homework #5','HW5Title'),
    P('Professor-aligned results and Python implementation. Report values are taken from the actual executed notebook.'),
    P('Question 1 — Two-epoch tanh neuron','HW5Head'),
    P('Given four pairs and starting weights (-1,1). The source specifies lambda=2 but omits the learning rate and update schedule. This solution explicitly uses the bipolar-sigmoid parameterisation tanh(lambda*z/2) and online gradient descent with eta=0.1.'),
    eq_image(r'E=\frac{1}{2}(y-\hat y)^2', 'HW5_E1.png'),
    eq_image(r'\mathbf{w}\leftarrow\mathbf{w}+\eta(y-\hat{y})(1-\hat{y}^2)\mathbf{x}','HW5_E2.png'),
    P(f'After epoch 1: w=({trace_q1.iloc[3].w1:.9f}, {trace_q1.iloc[3].w2:.9f}).'),
    P(f'After epoch 2: w=({w_q1[0]:.9f}, {w_q1[1]:.9f}). Training accuracy: {100*acc_q1:.2f}% (3/4). These numbers depend on the explicitly selected learning rate.'),
    P('Eight sample updates (weights shown after each sample):'),
]
for _,row in trace_q1.iterrows():
    story.append(P(f'Epoch {int(row.epoch)}, example {int(row["sample"])}: '
                   f'z={row.z:.6f}, output={row.tanh:.6f}, '
                   f'w=({row.w1:.9f}, {row.w2:.9f}).','HW5Small'))
story += [
    P('Question 2 — Three fully connected layers','HW5Head'),
    P('Fashion-MNIST: 54,000 training, 6,000 validation, 10,000 independent test images; pixel intensities scaled to [0,1]. The model has three Dense layers, 784-128(ReLU)-64(ReLU)-10(Softmax), and 109,386 trainable parameters.'),
    P('Loss: sparse categorical cross-entropy. Optimiser: Adam, initial learning rate 0.001. Batch size 128, 12 epochs, He-normal hidden-layer initialisation, and no Dropout.'),
    P(f'Test accuracy: <b>{100*accuracy_3:.2f}%</b> '
      f'({int(np.trace(cm_3))}/10000); test loss: {loss_3:.6f}.'),
    add_figure('HW5_CM_3Dense.png',width=143*mm),
    P('Question 3 — Five fully connected layers and GPU comparison','HW5Head'),
    P('Five Dense layers: 784-128(ReLU)-96(ReLU)-64(ReLU)-32(ReLU)-10(Softmax); 121,482 parameters. All training settings and datasets are identical to Q2.'),
    P(f'Test accuracy: <b>{100*accuracy_5:.2f}%</b> '
      f'({int(np.trace(cm_5))}/10000); test loss: {loss_5:.6f}. '
      f'Change versus 3 layers: {accuracy_diff_pp:+.3f} percentage points.'),
    P(accuracy_conclusion),
    P(hardware_sentence),
    add_figure('HW5_CM_5Dense.png',width=143*mm),
    add_figure('HW5_Accuracy_Comparison.png',width=145*mm),
    PageBreak(),
    P('Appendix — Complete Python Source Code','HW5Head'),
    P('The following reproduces Q1, Q2 and Q3. It includes loading Fashion-MNIST, network definitions, training, test evaluation, confusion matrices and a controlled CPU/GPU benchmark.','HW5Small'),
]

# Source code is appended with preserved indentation in a monospaced font.
# Split into manageable blocks so it cannot disappear if a block exceeds a page.
line_width=92
wrapped_lines=[]
for line in EXPERIMENT_SOURCE.splitlines():
    if len(line)>line_width:
        leading=len(line)-len(line.lstrip(' '))
        wrapped_lines.extend(textwrap.wrap(line,width=line_width,subsequent_indent=' '*(leading+4),
                                           break_long_words=True,break_on_hyphens=False))
    else:
        wrapped_lines.append(line)
for start in range(0,len(wrapped_lines),42):
    block='\n'.join(wrapped_lines[start:start+42])
    story.append(Preformatted(block,ParagraphStyle(name=f'Code_{start}',
                              fontName='Courier',fontSize=6.2,leading=8.2)))

def footer(canvas,doc):
    canvas.saveState()
    canvas.setFont('Helvetica',8)
    canvas.drawRightString(A4[0]-16*mm,8*mm,f'Page {doc.page}')
    canvas.restoreState()

doc=SimpleDocTemplate(REPORT_PATH,pagesize=A4,
                      leftMargin=16*mm,rightMargin=16*mm,
                      topMargin=16*mm,bottomMargin=17*mm)
doc.build(story,onFirstPage=footer,onLaterPages=footer)
print('Created professor-required single PDF:',REPORT_PATH)

# In Colab, offer a direct file download.
try:
    from google.colab import files
    files.download(REPORT_PATH)
except ImportError:
    pass